# BioFlowSim: Simple Flow Demonstration

This notebook demonstrates particle-flow simulation, trajectory visualization, synthetic image generation, imaging artifacts, velocity analysis, and video export.

All results are synthetic and intended for testing and experimentation.

In [ ]:
%pip install numpy matplotlib opencv-python

## Imports

Import the libraries and functions required for the simulation and visualization.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from simulation import FlowSimulator
from imaging import (
    render_frame,
    add_gaussian_noise,
    add_speckle,
    add_motion_blur,
    adjust_lighting,
    generate_video,
)

## 1. Configure the Simulation

Define the image dimensions, particle count, number of frames, and velocity field.

In [ ]:
WIDTH = 320
HEIGHT = 160
NUM_PARTICLES = 120
NUM_FRAMES = 100

def parabolic_velocity(x, y):
    """Return a simple horizontal velocity profile."""
    normalized_y = (y - HEIGHT / 2) / (HEIGHT / 2)
    vx = 4.0 * (1.0 - normalized_y**2)
    vy = 0.0
    return np.array([vx, vy])

## 2. Run the Simulation

Generate particle trajectories and velocity information using the flow simulator.

In [ ]:
simulator = FlowSimulator(
    num_particles=NUM_PARTICLES,
    frames=NUM_FRAMES,
    width=WIDTH,
    height=HEIGHT,
    diffusion=0.1,
    velocity_field=parabolic_velocity,
    random_seed=42,
)

trajectories, velocities, collisions = simulator.run()

print("Trajectories shape:", np.shape(trajectories))
print("Velocities shape:", np.shape(velocities))
print("Collisions:", collisions)

## 3. Visualize Particle Trajectories

Plot particle paths across the simulated frames to observe their movement.

In [ ]:
plt.figure(figsize=(10, 4))

for particle_idx in range(min(30, NUM_PARTICLES)):
    particle_path = trajectories[:, particle_idx, :]
    plt.plot(
        particle_path[:, 0],
        particle_path[:, 1],
        linewidth=0.8,
    )

plt.xlim(0, WIDTH)
plt.ylim(HEIGHT, 0)
plt.xlabel("X Position (pixels)")
plt.ylabel("Y Position (pixels)")
plt.title("Simulated Particle Trajectories")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Render a Synthetic Frame

Render one simulated time step as an image.

In [ ]:
FRAME_INDEX = min(50, NUM_FRAMES - 1)

frame = render_frame(
    trajectories[FRAME_INDEX],
    width=WIDTH,
    height=HEIGHT,
    particle_radius=2,
)

plt.figure(figsize=(10, 4))
plt.imshow(frame, cmap="gray")
plt.title(f"Simulated Flow: Frame {FRAME_INDEX}")
plt.axis("off")
plt.show()

## 5. Simulate Imaging Artifacts

Compare the original image with versions containing noise, speckle, motion blur, and lighting changes.

In [ ]:
gaussian_frame = add_gaussian_noise(frame, sigma=10)
speckle_frame = add_speckle(frame, intensity=0.2)
blurred_frame = add_motion_blur(frame, ksize=5, angle=0)
adjusted_frame = adjust_lighting(frame, factor=1.3)

images = [
    ("Original", frame),
    ("Gaussian Noise", gaussian_frame),
    ("Speckle", speckle_frame),
    ("Motion Blur", blurred_frame),
    ("Adjusted Lighting", adjusted_frame),
]

fig, axes = plt.subplots(1, len(images), figsize=(16, 4))

for ax, (title, image) in zip(axes, images):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Analyze Particle Velocities

Plot the average horizontal and vertical velocities over time.

In [ ]:
velocities = np.asarray(velocities)

mean_vx = velocities[:, :, 0].mean(axis=1)
mean_vy = velocities[:, :, 1].mean(axis=1)

plt.figure(figsize=(10, 4))
plt.plot(mean_vx, label="Mean Horizontal Velocity")
plt.plot(mean_vy, label="Mean Vertical Velocity")
plt.xlabel("Frame")
plt.ylabel("Velocity")
plt.title("Mean Particle Velocity Over Time")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Export a Synthetic Video

Export the simulated particle trajectories as a video for further analysis.

In [ ]:
os.makedirs("outputs", exist_ok=True)

generate_video(
    trajectories,
    output_path="outputs/particle_flow.avi",
    fps=20,
    width=WIDTH,
    height=HEIGHT,
    particle_radius=2,
    noise_type="gaussian",
    noise_level=10,
    speckle_intensity=0.2,
)

print("Video export attempted: outputs/particle_flow.avi")

## 8. Summary

This demonstration explored:

- Particle trajectories in a simulated flow field
- Synthetic image rendering
- Gaussian noise, speckle, motion blur, and lighting variations
- Average particle velocity over time
- Video export for downstream analysis

These synthetic outputs can support controlled experiments with image-processing
and machine-learning methods. They should be validated against real imaging data
before being used to support experimental conclusions.